这里说频率是指一个机场航班出现的频率

## 加载数据

数据已经预处理过了

In [1]:
import pandas as pd

# 加载数据
data = pd.read_csv('./pre_2023-2024_with_comp_train.csv', dtype={'flt_no': str})

# 查看前几行数据，确保加载成功
# 显示前两行数据以确保正确加载
print(data.shape)
print(data.head(5))
print(data.tail(5))

(1281375, 20)
  flt_no  cap aircraft  legs  leg_no  duration  pax    a    b    c  \
0   7558  110      195     1       1      1.30   97  AAT  URC  NaN   
1   7470  110      195     1       1      1.25   67  ACF  URC  NaN   
2   769R    0      190     3       3      4.75   22  ACF  TLQ  XIY   
3   5248  162      320     1       1      1.83   55  AKA  HGH  NaN   
4   6250  167      320     1       1      3.98  166  AKU  CGO  NaN   

    unit_price  competitor_price  year  month  day  weekday  hour  minute  \
0   470.474227        -68.798500  2023      1    1        6    14      35   
1   454.925373       -123.872095  2023      1    1        6    22      40   
2  1177.818182          0.000000  2023      1    1        6    18       0   
3   669.090909          0.000000  2023      1    1        6    12      55   
4  1794.783133        -65.028989  2023      1    1        6    13      10   

  from   to  
0  AAT  URC  
1  ACF  URC  
2  ACF  XIY  
3  AKA  HGH  
4  AKU  CGO  
        flt_no  ca

In [2]:
# 检查标准化后的统计信息
print("\n标准化后的统计信息：")
print(data['pax'].describe())


标准化后的统计信息：
count    1.281375e+06
mean     1.222024e+02
std      5.272962e+01
min      2.000000e+01
25%      8.300000e+01
50%      1.280000e+02
75%      1.590000e+02
max      3.110000e+02
Name: pax, dtype: float64


## 编码分类变量

### 新增城市标签

In [3]:
import json
# 加载字典
with open('../../my/encoder/city_labels_航班频率加权图标签.json', 'r') as file:
    city_labels_loaded = json.load(file)

print("加载的字典：", city_labels_loaded)

加载的字典： {'AAT': 2, 'CKG': 0, 'PEK': 0, 'TCG': 2, 'URC': 2, 'XIY': 0, 'YIN': 2, 'ACF': 2, 'HMI': 2, 'SHF': 2, 'TLQ': 2, 'ACX': 1, 'CGO': 0, 'KMG': 0, 'AEB': 1, 'CAN': 0, 'HGH': 0, 'KWL': 0, 'TFU': 0, 'AKA': 1, 'HAK': 0, 'TSN': 0, 'UYN': 0, 'AKU': 2, 'FOC': 0, 'HTN': 2, 'AOG': 0, 'TNA': 0, 'AQG': 0, 'KWE': 0, 'NGB': 0, 'TAO': 0, 'XMN': 0, 'AVA': 0, 'LLB': 0, 'BAR': 1, 'HRB': 0, 'JMJ': 1, 'KHN': 0, 'LHW': 0, 'SJW': 0, 'SZX': 0, 'BAV': 0, 'HLD': 1, 'TGO': 1, 'XIL': 1, 'BFJ': 0, 'NNG': 0, 'BHY': 0, 'HFE': 0, 'JNG': 0, 'NAO': 2, 'WUH': 0, 'XUZ': 0, 'YIH': 0, 'BPE': 0, 'CSX': 0, 'BPL': 2, 'BPX': 0, 'BSD': 1, 'BZX': 0, 'LJG': 0, 'NKG': 0, 'CGQ': 0, 'DLC': 0, 'DOY': 1, 'HDG': 0, 'HET': 0, 'HSN': 0, 'HZG': 2, 'INC': 0, 'JIQ': 0, 'PKX': 0, 'PVG': 0, 'SHA': 0, 'SHE': 0, 'SQJ': 0, 'SYX': 0, 'TEN': 0, 'TYN': 0, 'WEF': 0, 'WUA': 1, 'YIC': 0, 'ZQZ': 0, 'CGD': 0, 'JHG': 0, 'CIF': 0, 'DLU': 1, 'ENH': 0, 'GOQ': 2, 'HNY': 0, 'HUZ': 0, 'JGS': 0, 'JJN': 0, 'JNZ': 1, 'KCA': 2, 'KHG': 2, 'KOW': 0, 'KRL': 2, 'L

In [4]:
# 使用 map 对 'a', 'b', 'c', 'from', 'to' 列进行标签化，新增对应的标签列
data['a_label'] = data['a'].map(city_labels_loaded)
data['b_label'] = data['b'].map(city_labels_loaded)
data['c_label'] = data['c'].map(city_labels_loaded)
data['from_label'] = data['from'].map(city_labels_loaded)
data['to_label'] = data['to'].map(city_labels_loaded)

### 新增城市二维嵌入

In [5]:
import json

# 加载字典
with open('../../my/encoder/城市嵌入编码_航班频率加权图.json', 'r') as file:
    city_embeddings = json.load(file)

print("加载的字典：", city_embeddings)


加载的字典： {'AAT': [1.3597848415374756, 0.6660159826278687], 'CKG': [-0.0919976532459259, 0.6568402051925659], 'PEK': [-0.2624581456184387, 0.18506205081939697], 'TCG': [1.3452502489089966, 0.9850115776062012], 'URC': [1.191598653793335, 1.0679560899734497], 'XIY': [-0.5777866840362549, 0.9855008721351624], 'YIN': [1.9014595746994019, 0.7400669455528259], 'ACF': [1.4711376428604126, 0.9144527316093445], 'HMI': [0.39429816603660583, 0.8812238574028015], 'SHF': [1.0217688083648682, 0.19476288557052612], 'TLQ': [0.6065776944160461, 0.4862461984157562], 'ACX': [0.7235668301582336, -0.6459048390388489], 'CGO': [0.25691214203834534, 0.8677778244018555], 'KMG': [-0.3540305197238922, -0.0116242291405797], 'AEB': [0.7109003663063049, -1.2106705904006958], 'CAN': [-0.09886128455400467, 0.5134572982788086], 'HGH': [-0.2080717533826828, 0.4809180498123169], 'KWL': [-0.9176110625267029, 0.5989621877670288], 'TFU': [0.17415915429592133, 0.24932153522968292], 'AKA': [-0.07954857498407364, -0.708072543144

In [6]:
import pandas as pd
import numpy as np

# 将 city_embeddings 转换为 DataFrame
embedding_df = pd.DataFrame.from_dict(city_embeddings, orient='index', columns=['embedding_1', 'embedding_2'])
embedding_df.index.name = 'city'

# 用 'a', 'b', 'c', 'from', 'to' 字段与 embedding_df 合并
data = data.merge(embedding_df, left_on='a', right_index=True, how='left')
data.rename(columns={'embedding_1': 'a_embedding_1', 'embedding_2': 'a_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='b', right_index=True, how='left')
data.rename(columns={'embedding_1': 'b_embedding_1', 'embedding_2': 'b_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='c', right_index=True, how='left')
data.rename(columns={'embedding_1': 'c_embedding_1', 'embedding_2': 'c_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='from', right_index=True, how='left')
data.rename(columns={'embedding_1': 'from_embedding_1', 'embedding_2': 'from_embedding_2'}, inplace=True)

data = data.merge(embedding_df, left_on='to', right_index=True, how='left')
data.rename(columns={'embedding_1': 'to_embedding_1', 'embedding_2': 'to_embedding_2'}, inplace=True)

# 查看添加的新列
print(data[['a_embedding_1', 'a_embedding_2', 'b_embedding_1', 'b_embedding_2', 'c_embedding_1', 'c_embedding_2', 'from_embedding_1', 'from_embedding_2', 'to_embedding_1', 'to_embedding_2']])


         a_embedding_1  a_embedding_2  b_embedding_1  b_embedding_2  \
0             1.359785       0.666016       1.191599       1.067956   
1             1.471138       0.914453       1.191599       1.067956   
2             1.471138       0.914453       0.606578       0.486246   
3            -0.079549      -0.708073      -0.208072       0.480918   
4             1.652670       0.735235       0.256912       0.867778   
...                ...            ...            ...            ...   
1281370      -0.410526       1.041433      -0.637159       0.892440   
1281371      -0.410526       1.041433      -0.208072       0.480918   
1281372      -0.410526       1.041433      -0.272586       0.754549   
1281373      -0.410526       1.041433      -0.259620       0.789916   
1281374      -0.410526       1.041433      -0.577787       0.985501   

         c_embedding_1  c_embedding_2  from_embedding_1  from_embedding_2  \
0                  NaN            NaN          1.359785          0.666

### 频率编码

使用 json 保存和加载 city_map

In [7]:
# 加载city_map
with open('../../my/encoder/city_map_频率编码.json', 'r') as f:
    city_map = json.load(f)

# 使用 city_map 替换指定列的值
columns_to_replace = ['a', 'b', 'c', 'from', 'to']

# 遍历指定列并直接用 map 映射
for col in columns_to_replace:
    data[col] = data[col].map(city_map)


print(data)

        flt_no  cap aircraft  legs  leg_no  duration  pax      a       b  \
0         7558  110      195     1       1      1.30   97   1802  119514   
1         7470  110      195     1       1      1.25   67   1869  119514   
2         769R    0      190     3       3      4.75   22   1869     686   
3         5248  162      320     1       1      1.83   55    958   95607   
4         6250  167      320     1       1      3.98  166   5087  115275   
...        ...  ...      ...   ...     ...       ...  ...    ...     ...   
1281370   7020  167      7MX     1       1      1.30  133  29660  201975   
1281371   7468  161      738     1       1      1.82  143  29660   95607   
1281372   7632  161      738     1       1      2.62  139  29660   33132   
1281373   7830  161      738     3       1      1.63  131  29660   52213   
1281374   7522  161      738     1       1      2.50  140  29660  168047   

                c  ...  a_embedding_1  a_embedding_2  b_embedding_1  \
0             Na

### 'flt_no', 'bd_type', 'aircraft'编码

In [8]:
import joblib
from sklearn.preprocessing import LabelEncoder
import os

# 定义需要编码的分类特征
# categorical_columns = ['flt_no', 'bd_type', 'aircraft']
categorical_columns = ['flt_no', 'aircraft']

# 从保存的文件中加载编码器并应用到data
for col in categorical_columns:
    # 加载编码器
    encoder_path = os.path.join('../../my/encoder/', f"{col}_encoder_all.pkl")
    le = joblib.load(encoder_path)
    
    try:
        # 对data进行转换
        data[col] = le.transform(data[col])
        print(f"{col}列编码完成")
    except ValueError as e:
        # 如果遇到新的类别，打印错误信息
        print(f"{col}列编码出错: {str(e)}")
        # 找出新的类别
        new_categories = set(data[col]) - set(le.classes_)
        print(f"{col}列中的新类别: {new_categories}")

# 查看编码后的结果
print("\n编码后的前几行数据：")
print(data[categorical_columns].head())

flt_no列编码完成
aircraft列编码完成

编码后的前几行数据：
   flt_no  aircraft
0    2846         2
1    2723         2
2    3055         0
3     622         8
4    1443         8


## 特征和目标分离
我们要预测的是pax字段，其他字段作为特征。

In [9]:
# 特征列
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft',  'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','holiday', 'hour', 'minute', 'second', 'from', 'to','unit_price']]
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price']]
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label']]

# 有abc，有标签，有嵌入
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]
# X = data[['flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]
X = data[['flt_no', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'from', 'to','unit_price','competitor_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]

# 删除了abc，但有标签，有嵌入
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label','a_embedding_1' , 'a_embedding_2' , 'b_embedding_1','b_embedding_2' , 'c_embedding_1' , 'c_embedding_2' , 'from_embedding_1','from_embedding_2' , 'to_embedding_1' , 'to_embedding_2']]

# 目标列
y = data['pax']

### 对x进行标准化

In [10]:
from sklearn.preprocessing import StandardScaler

# 对所有特征进行标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 将标准化后的数据转回 DataFrame 格式
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 查看结果
pd.set_option('display.max_columns', None)  # 显示所有列
print(X_scaled.head(1))


# 保存 scaler
joblib.dump(scaler, '../../my/encoder/standard_scaler_x.pkl')
print("x的标准化器已保存为 standard_scaler_x.pkl")

     flt_no       cap  aircraft     legs    leg_no  duration         a  \
0  0.457056 -0.549994 -1.370183 -0.84848 -0.557563 -1.228429 -1.447827   

          b   c      year     month       day   weekday      hour    minute  \
0  1.000343 NaN -0.820235 -1.424451 -1.683372  1.479114 -0.001592  0.480775   

       from        to  unit_price  competitor_price   a_label   b_label  \
0 -1.349091  0.751312   -0.566373         -0.358995  3.457104  3.961738   

   c_label  from_label  to_label  a_embedding_1  a_embedding_2  b_embedding_1  \
0      NaN    3.636126  3.634265       2.916052       0.246584       2.963021   

   b_embedding_2  c_embedding_1  c_embedding_2  from_embedding_1  \
0       1.270389            NaN            NaN          3.058969   

   from_embedding_2  to_embedding_1  to_embedding_2  
0          0.278792        2.731207        1.220578  
x的标准化器已保存为 standard_scaler_x.pkl


### 对y进行标准化

In [11]:
# 对目标列 y 进行标准化
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))  # 将 y 转换为 2D 数组进行标准化

# 转换回 DataFrame 格式
y_scaled = pd.DataFrame(y_scaled, columns=['pax_scaled'])

# 查看标准化后的 y
print(y_scaled.head())

# 保存 y 的 scaler
joblib.dump(scaler_y, '../../my/encoder/standard_scaler_y.pkl')
print("y的标准化器已保存为 standard_scaler_y.pkl")

   pax_scaled
0   -0.477956
1   -1.046896
2   -1.900307
3   -1.274472
4    0.830607
y的标准化器已保存为 standard_scaler_y.pkl


## 训练XGBoost模型

In [12]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 自定义 SMAPE 函数
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    return np.mean(diff / denominator) * 100

# 自定义评估函数
def smape_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    smape_value = smape(y_true, y_pred)
    return 'SMAPE', smape_value

# 数据划分
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 输出数据集大小
print(f'训练集大小: {X_train.shape[0]}')
print(f'验证集大小: {X_val.shape[0]}')
print(f'测试集大小: {X_test.shape[0]}')

# 转换为 DMatrix 格式
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# 设置参数
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.1,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree':0.7,
    'alpha': 1,
    "lambda":1
}

# 设置评估集
evals = [(dtrain, 'train'), (dval, 'validation')]

# 训练模型，使用自定义评估指标
model = xgb.train(
    params,
    dtrain,
    num_boost_round=110,
    evals=evals,
    early_stopping_rounds=10,
    custom_metric=smape_eval,  # 使用 custom_metric 参数
    verbose_eval=10 #隔多少轮显示一次
)

# 预测测试集
y_pred = model.predict(dtest)

# # 测试集 SMAPE 评估
# test_smape = smape(y_test, y_pred)
# print(f'SMAPE on Test Set: {test_smape:.2f}%')

# # 测试集 MSE 评估
# mse = mean_squared_error(y_test, y_pred)
# print(f'Mean Squared Error on Test Set: {mse}')

训练集大小: 1025100
验证集大小: 128137
测试集大小: 128138
[0]	train-rmse:0.94412	train-SMAPE:168.81982	validation-rmse:0.94353	validation-SMAPE:168.69972
[10]	train-rmse:0.60104	train-SMAPE:91.95204	validation-rmse:0.60127	validation-SMAPE:92.00900
[20]	train-rmse:0.51121	train-SMAPE:79.38253	validation-rmse:0.51212	validation-SMAPE:79.43080
[30]	train-rmse:0.48632	train-SMAPE:75.49502	validation-rmse:0.48773	validation-SMAPE:75.60666
[40]	train-rmse:0.47457	train-SMAPE:73.82143	validation-rmse:0.47649	validation-SMAPE:74.00060
[50]	train-rmse:0.46629	train-SMAPE:72.60324	validation-rmse:0.46860	validation-SMAPE:72.83056
[60]	train-rmse:0.46038	train-SMAPE:71.83760	validation-rmse:0.46298	validation-SMAPE:72.11227
[70]	train-rmse:0.45524	train-SMAPE:71.15326	validation-rmse:0.45797	validation-SMAPE:71.43560
[80]	train-rmse:0.45172	train-SMAPE:70.70318	validation-rmse:0.45470	validation-SMAPE:70.98858
[90]	train-rmse:0.44812	train-SMAPE:70.24226	validation-rmse:0.45139	validation-SMAPE:70.54304
[100]	

In [13]:
y_pred

array([ 0.50050807,  0.88549006,  0.35104975, ...,  0.31402552,
        0.3938404 , -0.10671932], dtype=float32)

In [14]:
# 反标准化 y_test
y_test_original = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

# 反标准化预测结果 y_pred
y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1))

In [15]:
y_test_original

array([[161.],
       [170.],
       [152.],
       ...,
       [151.],
       [151.],
       [ 95.]])

In [16]:
y_pred_original

array([[148.59401],
       [168.89395],
       [140.71313],
       ...,
       [138.76086],
       [142.96947],
       [116.57515]], dtype=float32)

In [17]:
# 正确的写法
test_results = list(zip(y_test_original[:100], y_pred_original[:100]))  # 真实值和预测值

print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results[:40]):
    # 如果是多维数组，使用 .item() 转换为标量
    true_value = true_value.item() if isinstance(true_value, np.ndarray) else true_value
    pred_value = pred_value.item() if isinstance(pred_value, np.ndarray) else pred_value
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")



20条测试结果（真实值 vs 预测值）:
第1条: 真实值=161.0, 预测值=148.59
第2条: 真实值=170.0, 预测值=168.89
第3条: 真实值=152.0, 预测值=140.71
第4条: 真实值=106.0, 预测值=93.05
第5条: 真实值=156.0, 预测值=162.32
第6条: 真实值=36.0, 预测值=64.77
第7条: 真实值=166.0, 预测值=138.51
第8条: 真实值=44.0, 预测值=49.56
第9条: 真实值=46.0, 预测值=112.68
第10条: 真实值=86.0, 预测值=138.41
第11条: 真实值=179.0, 预测值=181.20
第12条: 真实值=171.0, 预测值=153.51
第13条: 真实值=93.0, 预测值=63.56
第14条: 真实值=90.0, 预测值=139.16
第15条: 真实值=160.0, 预测值=169.39
第16条: 真实值=99.0, 预测值=118.31
第17条: 真实值=96.0, 预测值=131.55
第18条: 真实值=145.0, 预测值=131.27
第19条: 真实值=229.0, 预测值=242.65
第20条: 真实值=177.0, 预测值=149.24
第21条: 真实值=172.0, 预测值=152.85
第22条: 真实值=217.0, 预测值=223.71
第23条: 真实值=38.0, 预测值=46.41
第24条: 真实值=146.0, 预测值=137.20
第25条: 真实值=181.0, 预测值=156.91
第26条: 真实值=138.0, 预测值=133.62
第27条: 真实值=93.0, 预测值=144.80
第28条: 真实值=104.0, 预测值=71.17
第29条: 真实值=184.0, 预测值=196.20
第30条: 真实值=162.0, 预测值=131.70
第31条: 真实值=149.0, 预测值=145.43
第32条: 真实值=180.0, 预测值=174.18
第33条: 真实值=242.0, 预测值=225.37
第34条: 真实值=104.0, 预测值=135.44
第35条: 真实值=109.0, 预测值=77.63
第36条: 真实值=95.0, 预测值=79.0

In [18]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_pred - y_true)
    
    # 避免除以零，将分母中为零的项替换为一个小值
    denominator = np.where(denominator == 0, 1e-8, denominator)
    
    smape = 100 * np.mean(diff / denominator)
    return smape


def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    
    # 避免除以零，将 y_true 中的零值替换为一个小值
    y_true = np.where(y_true == 0, 1e-8, y_true)
    
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape


# 假设 y_test 和 y_pred 已经是标准化反归一化后的数据
# 将其转换为一维数组以确保形状一致
y_test = np.array(y_test_original).ravel()
y_pred = np.array(y_pred_original).ravel()

# 评估指标
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = calculate_mape(y_test, y_pred)
smape = calculate_smape(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 打印结果
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')
print(f'R-squared (R²): {r2:.4f}')


Mean Squared Error (MSE): 552.8827
Root Mean Squared Error (RMSE): 23.5135
Mean Absolute Error (MAE): 18.0636
Mean Absolute Percentage Error (MAPE): 21.7906%
Symmetric Mean Absolute Percentage Error (SMAPE): 18.4504%
R-squared (R²): 0.8032


似乎对于较小值预测存在误差

## 保存模型

In [19]:
model.save_model("../../my/model/频率编码/归一化_xgboost_model_1000.json")
print("模型已保存为 xgboost_model_1000.json")

模型已保存为 xgboost_model_1000.json


## 超参数设置

## 不同特征重要程度测试

In [20]:
# import xgboost as xgb
# import matplotlib.pyplot as plt

# # 假设 model 是训练好的 XGBoost 模型
# xgb.plot_importance(model, importance_type='weight', title="Feature Importance (Weight)", height=0.5)
# plt.show()

# xgb.plot_importance(model, importance_type='gain', title="Feature Importance (Gain)", height=0.5)
# plt.show()

# xgb.plot_importance(model, importance_type='cover', title="Feature Importance (Cover)", height=0.5)
# plt.show()